# 第4章：分布式训练与效率优化

## 本章目标
- 理解 DDP (Distributed Data Parallel) 的原理
- 掌握混合精度训练 (FP16/BF16) 的实现
- 理解 Flash Attention 的加速原理
- 学会使用 gradient accumulation 模拟大 batch
- 计算 MFU (Model FLOPs Utilization)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## 训练效率优化速览

大模型训练的瓶颈通常不是计算（FLOPs），而是**显存**和**通信**。

关键优化手段：
- **混合精度**：用 FP16/BF16 做前向反向，减少显存和加速计算
- **Gradient Accumulation**：多次小 batch 的梯度累积，模拟大 batch
- **Flash Attention**：优化 attention 的显存访问模式，减少 HBM 读写
- **DDP**：多卡数据并行，梯度 all-reduce 同步

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention.

    与标准 Self-Attention 的区别：
    1. 多头并行计算 (n_head 个独立的 attention head)
    2. Causal mask：只关注当前位置及之前的 token
    """
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)  # Q, K, V 合并计算
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        # causal mask: 下三角矩阵
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    """Feed-forward network with GELU activation.

    标准做法：d_model → 4 × d_model → d_model
    GPT-2 使用 GELU 而不是 ReLU。
    """
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    """Pre-norm Transformer Block.

    GPT 使用 Pre-LN（LayerNorm 在 attention/MLP 之前），
    而原始 Transformer 使用 Post-LN。
    """
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # residual connection
        x = x + self.mlp(self.ln_2(x))     # residual connection
        return x


class GPTConfig:
    """GPT-2 的配置参数"""
    def __init__(self, vocab_size=50304, block_size=1024,
                 n_layer=12, n_head=12, n_embd=768, dropout=0.1):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.block_size = config.block_size
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)    # token embedding
        self.wpe = nn.Embedding(config.block_size, config.n_embd)    # position embedding
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config.n_embd, config.n_head,
                                       config.block_size, config.dropout)
                                 for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying: embedding 和 output head 共享权重
        self.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
def measure_memory(model, label):
    """测量模型前向+反向的峰值显存"""
    torch.cuda.reset_peak_memory_stats()
    x = torch.randint(0, 65, (4, 128), device=model.device)
    _, loss = model(x, x)
    loss.backward()
    peak_mem = torch.cuda.max_memory_allocated() / 1e6
    print(f"{label}: peak memory = {peak_mem:.1f} MB")

if torch.cuda.is_available():
    device = "cuda"
    # FP32
    model_fp32 = GPT(GPTConfig(vocab_size=65, block_size=128, n_layer=4, n_head=4, n_embd=128)).to(device)
    measure_memory(model_fp32, "FP32")
    del model_fp32
    torch.cuda.empty_cache()

    # FP16
    model_fp16 = GPT(GPTConfig(vocab_size=65, block_size=128, n_layer=4, n_head=4, n_embd=128)).half().to(device)
    measure_memory(model_fp16, "FP16")
else:
    print("需要 GPU 才能运行此 cell。Colab T4 推荐运行。")

In [ ]:
if torch.cuda.is_available():
    from torch.cuda.amp import autocast, GradScaler

    model = GPT(GPTConfig(vocab_size=65, block_size=128, n_layer=4, n_head=4, n_embd=128)).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scaler = GradScaler()

    x = torch.randint(0, 65, (4, 128)).cuda()
    with autocast():
        logits, loss = model(x, x)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    print("Mixed precision training step complete")
else:
    print("需要 GPU。AMP 的关键：autocast 自动选择 FP16/FP32，GradScaler 处理梯度下溢。")

In [ ]:
accumulation_steps = 4
micro_batch_size = 8
effective_batch_size = micro_batch_size * accumulation_steps
print(f"micro batch: {micro_batch_size}, accumulation: {accumulation_steps}")
print(f"effective batch size: {effective_batch_size}")
print()
print("Gradient Accumulation 流程：")
print("1. optimizer.zero_grad()")
for step in range(accumulation_steps):
    print(f"2. micro step {step+1}: forward + backward (loss /= accumulation_steps)")
print("3. optimizer.step()")

## Flash Attention

Flash Attention 的核心思想：**减少 HBM (High Bandwidth Memory) 的访问次数**。

标准 Attention 需要 O(N²) 的显存来存储 attention matrix。Flash Attention 通过：
1. 分块计算（tiling）—— 在 SRAM 中完成 attention 计算
2. 不显式存储完整的 attention matrix
3. IO-aware —— 优化计算和显存访问的平衡

结果：显存从 O(N²) 降到 O(N)，速度提升 2-4x。

In [ ]:
if torch.cuda.is_available():
    from torch.nn.functional import scaled_dot_product_attention

    q = torch.randn(2, 8, 128, 64, device="cuda", dtype=torch.float16)
    k = torch.randn(2, 8, 128, 64, device="cuda", dtype=torch.float16)
    v = torch.randn(2, 8, 128, 64, device="cuda", dtype=torch.float16)

    # Flash Attention (is_causal=True)
    with torch.backends.cuda.sdp_kernel(enable_flash=True):
        out_flash = scaled_dot_product_attention(q, k, v, is_causal=True)

    # 标准 attention (对比)
    out_standard = scaled_dot_product_attention(q, k, v, is_causal=True)

    print(f"Flash output shape: {out_flash.shape}")
    print(f"Max diff: {(out_flash - out_standard).abs().max().item():.6f}")
    print("Flash Attention 和标准实现结果一致，但速度更快、显存更省。")
else:
    print("需要 GPU（Colab T4）。PyTorch 2.0+ 自动使用 Flash Attention。")

In [ ]:
print("DDP (Distributed Data Parallel) 核心流程：")
print()
print("1. 每个 GPU 持有模型的完整副本")
print("2. 数据被分片到各个 GPU")
print("3. 每个 GPU 独立完成 forward + backward")
print("4. All-Reduce 同步所有 GPU 的梯度（取平均）")
print("5. 每个 GPU 独立更新参数（结果一致）")
print()
print("实际使用：")
print("  torchrun --nproc_per_node=4 train.py  # 4卡 DDP")
print()
print("关键注意：")
print("- 只有数据并行，模型在每个 GPU 上是完整的")
print("- 适合模型能放进单卡，但想加速训练的场景")
print("- 对于超大模型，需要 FSDP 或 Pipeline Parallelism")

In [ ]:
import time

def estimate_mfu(model, batch_size, seq_len, dt):
    """估算 Model FLOPs Utilization"""
    n_params = sum(p.numel() for p in model.parameters())
    flops_per_token = 6 * n_params  # forward ≈ 3N, backward ≈ 3N
    tokens_per_sec = batch_size * seq_len / dt
    achieved_tflops = flops_per_token * tokens_per_sec / 1e12
    hardware_tflops = 65.0  # T4 FP16 theoretical
    mfu = achieved_tflops / hardware_tflops * 100
    return mfu

if torch.cuda.is_available():
    config = GPTConfig(vocab_size=65, block_size=256, n_layer=6, n_head=6, n_embd=384)
    model = GPT(config).cuda()
    x = torch.randint(0, 65, (16, 256), device="cuda")

    # warmup
    for _ in range(3):
        _, loss = model(x, x)
        loss.backward()

    # measure
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(10):
        _, loss = model(x, x)
        loss.backward()
    torch.cuda.synchronize()
    t1 = time.time()

    mfu = estimate_mfu(model, 16, 256, (t1-t0)/10)
    print(f"MFU: {mfu:.1f}%")
    print(f"(T4 理论算力 65 TFLOPS, 实际利用 {mfu:.1f}%)")
else:
    print("需要 GPU 测量 MFU。Colab T4 上运行此 cell。")

## 练习

1. 对比不同模型大小的 MFU，观察是否小模型 MFU 更高（计算效率更好）
2. 修改 Flash Attention 的 seq_len，观察显存占用随序列长度的变化
3. 在双卡环境上运行 `torchrun --nproc_per_node=2`，体验 DDP 训练

## 延伸阅读

- [Flash Attention 论文](https://arxiv.org/abs/2205.14135)
- [PyTorch DDP Tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- [nanoGPT train.py](https://github.com/karpathy/nanoGPT/blob/master/train.py) — 完整的 DDP + AMP 实现
- [FSDP Tutorial](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)